In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
# 1. Carregar as tabelas Gold necessárias
fact_movies = spark.table("cinedata.gold.fact_movies_performance")
dim_movies = spark.table("cinedata.gold.dim_movies")
dim_genres = spark.table("cinedata.gold.dim_genres")
dim_people = spark.table("cinedata.gold.dim_people")
dim_companies = spark.table("cinedata.gold.dim_companies")
bridge_genre = spark.table("cinedata.gold.bridge_movie_genre")
bridge_person = spark.table("cinedata.gold.bridge_movie_person")
bridge_company = spark.table("cinedata.gold.bridge_movie_company")

In [0]:
# Obter a data de lançamento mais recente válida (ignorando nulos ou datas futuras em relação a hoje)
data_limite_base = (
    dim_movies
    .filter(F.col("data_lancamento").isNotNull() & (F.col("data_lancamento") <= F.current_date()))
    .agg(F.max("data_lancamento"))
    .collect()[0][0]
)
print(f"Data limite superior de referência para os recortes: {data_limite_base}")

Data limite superior de referência para os recortes: 2026-02-19


In [0]:
# ==============================================================================
# Pergunta 1: Qual é a receita total (em R$) somada de todos os filmes da base?
# ==============================================================================
print("\n--- PERGUNTA 1: Receita Total em R$ ---")
df_p1 = fact_movies.agg(F.sum("receita_brl").alias("receita_total_brl"))
display(df_p1)


--- PERGUNTA 1: Receita Total em R$ ---


receita_total_brl
833266912049.70


In [0]:
# ==============================================================================
# Pergunta 2: Quais são os 5 filmes com maior popularidade? Mostre título e valor.
# ==============================================================================
print("\n--- PERGUNTA 2: Top 5 Filmes por Popularidade ---")
df_p2 = (
    fact_movies
    .join(dim_movies, "sk_movie_id")
    .select("titulo", "popularidade")
    .orderBy(F.col("popularidade").desc_nulls_last())
    .limit(5)
)
display(df_p2)


--- PERGUNTA 2: Top 5 Filmes por Popularidade ---


titulo,popularidade
Asylum of the Devil,2.0093201020121E12
Konrad Mägi,2.019202E7
A Postcard from Pyongyang,2.0132017E7
"The psychopath, chronicle of an unsolved case",1.9851995E7
100 YEARS OF MEN IN LOVE: THE ACCIDENTAL COLLECTION,1.850195E7


In [0]:
# ==============================================================================
# Pergunta 3: Quantos filmes cada gênero possui? Liste do maior para o menor.
# ==============================================================================
print("\n--- PERGUNTA 3: Quantidade de Filmes por Gênero ---")
df_p3 = (
    bridge_genre
    .join(dim_genres, "sk_genre_id")
    .groupBy("nome_genero")
    .agg(F.count("sk_movie_id").alias("total_filmes"))
    .orderBy(F.col("total_filmes").desc())
)
display(df_p3)


--- PERGUNTA 3: Quantidade de Filmes por Gênero ---


nome_genero,total_filmes
Drama,32305
Documentary,19071
Comedy,18627
Thriller,10274
Horror,9727
Romance,7640
Action,6056
Crime,4739
Animation,4471
Science Fiction,3771


In [0]:
# ==============================================================================
# Pergunta 4: Para os 10 filmes de maior receita, mostre título, receita (US$ e R$) e o RANK().
# ==============================================================================
print("\n--- PERGUNTA 4: Top 10 Filmes por Receita (com RANK) ---")
window_receita = Window.orderBy(F.col("receita_usd").desc_nulls_last())

df_p4 = (
    fact_movies
    .join(dim_movies, "sk_movie_id")
    .select(
        "titulo", 
        "receita_usd", 
        "receita_brl", 
        F.rank().over(window_receita).alias("ranking_receita")
    )
    .orderBy("ranking_receita")
    .limit(10)
)
display(df_p4)


--- PERGUNTA 4: Top 10 Filmes por Receita (com RANK) ---


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


titulo,receita_usd,receita_brl,ranking_receita
Avengers: Endgame,2800000000.00,13913760000.00,1
Avatar: The Way of Water,2320250281.00,11529787696.35,2
AVENGERS: INFINITY WAR,2052415039.00,10198860811.80,3
spider-man: no way home,1921847111.00,9550042663.98,4
The Lion King,1663075401.00,8264154282.65,5
Top Gun: Maverick,1488732821.00,7397811134.11,6
Barbie,1428545028.00,7098725953.14,7
The Super Mario Bros. Movie,1355725263.00,6736869976.90,8
Black Panther,1349926083.00,6708052691.64,9
Star Wars: The Last Jedi,1332698830.00,6622447026.04,10


In [0]:
# ==============================================================================
# Pergunta 5: Qual ator teve a maior quantidade de participações nos últimos 2 anos?
# ==============================================================================
print("\n--- PERGUNTA 5: Ator com mais participações nos últimos 2 anos ---")
df_p5 = (
    bridge_person
    .join(dim_people, "sk_person_id")
    .join(dim_movies, "sk_movie_id")
    .filter(
        (F.col("tipo_pessoa") == "Ator") & 
        (F.col("data_lancamento") >= F.expr(f"date_sub('{data_limite_base}', 730)"))
    )
    .groupBy("nome_pessoa")
    .agg(F.count("sk_movie_id").alias("total_participacoes"))
    .orderBy(F.col("total_participacoes").desc())
    .limit(1)
)
display(df_p5)


--- PERGUNTA 5: Ator com mais participações nos últimos 2 anos ---


nome_pessoa,total_participacoes
Sterling L. Pope,12


In [0]:
# ==============================================================================
# Pergunta 6: Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos?
# ==============================================================================
print("\n--- PERGUNTA 6: Produtora com maior Lucro nos últimos 5 anos ---")
df_p6 = (
    bridge_company
    .join(dim_companies, "sk_company_id")
    .join(dim_movies, "sk_movie_id")
    .join(fact_movies, "sk_movie_id")
    .filter(
        F.col("data_lancamento") >= F.expr(f"date_sub('{data_limite_base}', 1825)")
    )
    .groupBy("nome_produtora")
    .agg(F.sum("lucro_usd").alias("lucro_total_usd"))
    .orderBy(F.col("lucro_total_usd").desc_nulls_last())
    .limit(1)
)
display(df_p6)


--- PERGUNTA 6: Produtora com maior Lucro nos últimos 5 anos ---


nome_produtora,lucro_total_usd
Universal Pictures,5860119834.00
